# Экономический рост и условия жизни
## Сравнение 2010 и 2019 годов по данным Всемирного банка

**Один вопрос:** У большинства экономик с ростом реального ВВП на душу населения с 2010 по 2019 год улучшились хотя бы три из четырёх выбранных социальных показателей?

**Критерий ответа:** Экономический рост означает, что реальный ВВП на душу в 2019 году строго выше уровня 2010 года. Социальные улучшения: увеличение ожидаемой продолжительности жизни, снижение младенческой смертности, снижение безработицы, увеличение доступа к электричеству. Равенство не считается улучшением. Ответ «да», если не менее трёх улучшений наблюдается более чем у 50% экономик с ростом ВВП; иначе «нет». Порог 3 из 4 — рабочее определение проекта, а не официальный индекс благополучия.

**Участники:** Гусейнов Елшан; Мария Москаленко.

**Запуск:** сохраните `economy_data.csv` рядом с блокнотом, откройте блокнот из этой папки и выполните все ячейки сверху вниз. Рабочая папка среды должна совпадать с папкой блокнота. Интернет не нужен. Последняя ячейка создаст `economy_results.txt`. В коде нет импортов.

## 1. Данные и пять показателей

Источник — World Development Indicators (World Bank). Получено 05.10.2026; поле lastupdated в ответах API: 2026-07-13. Берём 2010 и 2019 годы, чтобы рассмотреть изменения за девять лет до пандемии. Названия экономик сохранены на английском из источника. «Экономики» включают страны и отдельные территории, поэтому их число не равно числу суверенных государств.

| Поле в CSV | Код Всемирного банка | Значение и роль |
|---|---|---|
| gdp_pc | NY.GDP.PCAP.KD | ВВП на душу в постоянных долларах США 2015 г.; определяет наличие реального роста |
| life | SP.DYN.LE00.IN | Ожидаемая продолжительность жизни при рождении, лет; рост считается улучшением |
| infant | SP.DYN.IMRT.IN | Младенческая смертность на 1000 живорождений; снижение считается улучшением |
| unemployment | SL.UEM.TOTL.ZS | Безработица, % рабочей силы, модельная оценка МОТ; снижение считается улучшением |
| electricity | EG.ELC.ACCS.ZS | Доступ к электричеству, % населения; рост считается улучшением |

ВВП выделяет экономики с ростом выпуска на человека. Четыре остальных показателя проверяют изменения здоровья, рынка труда и базовой инфраструктуры, поэтому каждый из пяти показателей участвует в ответе. Два показателя здоровья отражают разные стороны, но связаны между собой: четыре критерия не являются независимыми измерениями.

**Подготовка файла.** Из API взяты все индивидуальные экономики, присутствующие в выгрузке WDI. Региональные и доходные агрегаты исключены по справочнику country (region.value = Aggregates). Оставлены только 2010 и 2019 годы; JSON преобразован в CSV с разделителем `;`. Изменены только формат и названия колонок: исходные числа не округлены, null записан пустым полем. Набор содержит 217 экономик и 434 строки. Предварительно по результатам изменений экономики не отбирались. В названиях нет `;` и переводов строки, поэтому split подходит.

**Правило пропусков:** анализируем одну и ту же выборку с полными значениями всех пяти показателей в обоих годах. Пустое поле не заменяется нулём. Код сам определяет исключённые экономики и печатает, каких значений не хватает. Все исходные строки остаются в data.

**Ссылки:** https://data.worldbank.org/indicator/NY.GDP.PCAP.KD ; https://data.worldbank.org/indicator/SP.DYN.LE00.IN ; https://data.worldbank.org/indicator/SP.DYN.IMRT.IN ; https://data.worldbank.org/indicator/SL.UEM.TOTL.ZS ; https://data.worldbank.org/indicator/EG.ELC.ACCS.ZS . Лицензия показателей: CC BY 4.0.

Шаблон API: `https://api.worldbank.org/v2/country/all/indicator/CODE?date=2010:2019&format=json&per_page=20000`, где CODE — код из таблицы. Справочник: `https://api.worldbank.org/v2/country?format=json&per_page=400`. Файл для запуска приложен, обращаться к API из блокнота не требуется.

## 2. Чтение файла и словарь

`data[код][год][показатель]` хранит числовое значение или пустую строку при пропуске. Ключ кода надёжнее названия: он короткий и уникальный в выгрузке. Год позволяет обращаться к двум срезам без поиска строк. `names[код]` хранит подпись экономики. Читаем встроенной open, разбираем построчно через strip и split.

In [1]:
# Словарь хранит все записи, включая строки с пропусками.
data = {}
names = {}
indicators = ['gdp_pc', 'life', 'infant', 'unemployment', 'electricity']
years = [2010, 2019]
with open('economy_data.csv', 'r', encoding='utf-8') as input_file:
    for line in input_file:
        parts = line.strip().split(';')
        if parts[0] != 'code':
            country = parts[0]
            year = int(parts[2])
            names[country] = parts[1]
            if country not in data:
                data[country] = {}
            data[country][year] = {}
            for i in range(len(indicators)):
                value = parts[i + 3]
                if value != '':
                    value = float(value)
                data[country][year][indicators[i]] = value
print('Экономик в файле:', len(data))
# Для двух лет используем одинаковую выборку с полными наблюдениями.
complete = []
excluded = {}
for country in data:
    missing = []
    for year in years:
        for indicator in indicators:
            if data[country][year][indicator] == '':
                missing.append(str(year) + ':' + indicator)
    if len(missing) == 0:
        complete.append(country)
    else:
        excluded[country] = ', '.join(missing)
print('Полных наблюдений за два года:', len(complete))
print('Исключено из расчётов:', len(excluded))
for country in excluded:
    print(country, names[country], excluded[country], sep=';')

Экономик в файле: 217
Полных наблюдений за два года: 174
Исключено из расчётов: 43
ABW;Aruba;2010:infant, 2010:unemployment, 2019:infant, 2019:unemployment
AND;Andorra;2010:unemployment, 2019:unemployment
ASM;American Samoa;2010:infant, 2010:unemployment, 2010:electricity, 2019:infant, 2019:unemployment, 2019:electricity
ATG;Antigua and Barbuda;2010:unemployment, 2019:unemployment
BMU;Bermuda;2010:infant, 2010:unemployment, 2019:infant, 2019:unemployment
CHI;Channel Islands;2010:infant, 2019:infant
CUW;Curacao;2010:infant, 2010:unemployment, 2019:infant, 2019:unemployment
CYM;Cayman Islands;2010:infant, 2010:unemployment, 2019:infant, 2019:unemployment
DJI;Djibouti;2010:gdp_pc
DMA;Dominica;2010:unemployment, 2019:unemployment
ERI;Eritrea;2019:gdp_pc
FRO;Faroe Islands;2010:infant, 2010:unemployment, 2019:infant, 2019:unemployment
FSM;Micronesia, Fed. Sts.;2010:unemployment, 2019:unemployment
GIB;Gibraltar;2010:gdp_pc, 2010:infant, 2010:unemployment, 2019:gdp_pc, 2019:infant, 2019:unempl

## 3. Пять разных описательных статистик

Для каждого показателя в каждом году считаем **среднее, медиану, минимум, максимум и размах**. Среднее = сумма / число наблюдений. Медиана — центральное значение отсортированного списка; если длина чётная — среднее двух центральных. Размах = максимум − минимум. Повторяющийся расчёт вынесен в собственную функцию.

`stats[год][показатель]` хранит список пяти статистик в указанном порядке. Все экономики имеют одинаковый вес: среднее не является средним для жителя мира. Округление применяется только при выводе.

In [2]:
# Одна функция используется для каждого показателя и каждого года.
def describe(values):
    ordered = sorted(values)
    count = len(ordered)
    mean = sum(values) / count
    if count % 2 == 1:
        median = ordered[count // 2]
    else:
        median = (ordered[count // 2 - 1] + ordered[count // 2]) / 2
    minimum = min(values)
    maximum = max(values)
    return [mean, median, minimum, maximum, maximum - minimum]

stats = {}
print('Год;Показатель;Среднее;Медиана;Минимум;Максимум;Размах')
for year in years:
    stats[year] = {}
    for indicator in indicators:
        values = []
        for country in complete:
            values.append(data[country][year][indicator])
        row = describe(values)
        stats[year][indicator] = row
        print(year, indicator, round(row[0], 3), round(row[1], 3), round(row[2], 3), round(row[3], 3), round(row[4], 3), sep=';')

Год;Показатель;Среднее;Медиана;Минимум;Максимум;Размах
2010;gdp_pc;12276.663;4941.147;253.309;106545.855;106292.546
2010;life;69.908;71.894;45.577;82.843;37.266
2010;infant;26.636;16.7;2.0;112.5;110.5
2010;unemployment;8.204;6.91;0.45;33.135;32.685
2010;electricity;77.786;97.45;5.3;100.0;94.7
2019;gdp_pc;13761.738;5757.949;270.411;107591.9;107321.488
2019;life;72.398;73.201;31.53;84.356;52.826
2019;infant;20.94;13.7;1.8;128.9;127.1
2019;unemployment;7.112;5.018;0.1;28.468;28.368
2019;electricity;84.61;99.75;8.4;100.0;91.6


## 4. Изменения между двумя годами и крайние случаи

Для всех пяти показателей: Δ = значение 2019 − значение 2010. Для ВВП дополнительно: темп прироста за весь период = (ВВП 2019 / ВВП 2010 − 1) × 100%. Это не среднегодовой темп. Изменения безработицы и доступа к электричеству выражены в п.п.; изменения продолжительности жизни — в годах; смертности — в случаях на 1000 живорождений. Для каждого показателя отдельно называем минимум и максимум разности со знаком. При совпадениях выводим все экономики. Для младенческой смертности и безработицы минимальная разность означает наибольшее снижение показателя.

In [3]:
changes = {}
growth = {}
for country in complete:
    changes[country] = {}
    for indicator in indicators:
        changes[country][indicator] = data[country][2019][indicator] - data[country][2010][indicator]
    growth[country] = (data[country][2019]['gdp_pc'] / data[country][2010]['gdp_pc'] - 1) * 100
# Находим крайние разности отдельно в шкале каждого показателя.
extremes = {}
for indicator in indicators:
    values = []
    for country in complete:
        values.append(changes[country][indicator])
    lowest = min(values)
    highest = max(values)
    lowest_names = []
    highest_names = []
    for country in complete:
        if changes[country][indicator] == lowest:
            lowest_names.append(names[country])
        if changes[country][indicator] == highest:
            highest_names.append(names[country])
    extremes[indicator] = [lowest, lowest_names, highest, highest_names]
    print(indicator, 'Минимум:', round(lowest, 4), ', '.join(lowest_names))
    print(indicator, 'Максимум:', round(highest, 4), ', '.join(highest_names))
print('Код;Экономика;Δ ВВП на душу, долл. 2015;Прирост ВВП, %;Δ жизни, лет;Δ смертности на 1000;Δ безработицы, п.п.;Δ электричества, п.п.')
for country in complete:
    row = changes[country]
    print(country, names[country], round(row['gdp_pc'], 3), round(growth[country], 3), round(row['life'], 3), round(row['infant'], 3), round(row['unemployment'], 3), round(row['electricity'], 3), sep=';')

gdp_pc Минимум: -10841.4415 Qatar
gdp_pc Максимум: 29493.9894 Ireland
life Минимум: -15.341 Central African Republic
life Максимум: 18.751 Haiti
infant Минимум: -37.5 Somalia, Fed. Rep.
infant Максимум: 40.4 Central African Republic
unemployment Минимум: -15.721 North Macedonia
unemployment Максимум: 5.829 Iraq
electricity Минимум: -13.4 Libya
electricity Максимум: 55.0 Afghanistan
Код;Экономика;Δ ВВП на душу, долл. 2015;Прирост ВВП, %;Δ жизни, лет;Δ смертности на 1000;Δ безработицы, п.п.;Δ электричества, п.п.
AFG;Afghanistan;14.991;2.761;2.239;-20.3;3.378;55.0
AGO;Angola;-435.192;-12.291;5.809;-15.6;-0.098;10.6
ALB;Albania;1492.612;41.695;1.053;-1.6;-2.62;0.4
ARE;United Arab Emirates;1139.607;2.692;0.71;-2.0;-0.467;0.0
ARG;Argentina;-680.758;-5.085;1.167;-3.5;2.129;1.2
ARM;Armenia;1547.377;55.341;3.049;-5.6;-1.219;0.2
AUS;Australia;5412.95;10.067;1.205;-0.8;-0.071;0.0
AUT;Austria;3515.155;8.168;1.315;-0.7;-0.323;0.0
AZE;Azerbaijan;46.843;0.884;2.981;-13.2;-0.63;0.1
BDI;Burundi;17.102;

## 5. Проверка вопроса

Для каждой экономики отдельно считаем число социальных улучшений. Рассматриваем только экономики с ростом реального ВВП на душу; знаменатель доли — их количество, а не все 174 экономики. Порог 3 из 4 позволяет не требовать роста всех показателей: например, доступ к электричеству уже может составлять 100%. Сохранение 100% при этом не считается новым улучшением. Это прозрачное, но условное правило, а не полная оценка уровня жизни.

In [4]:
# Для жизни и электричества улучшение — рост; для остальных — снижение.
directions = {'life': 1, 'infant': -1, 'unemployment': -1, 'electricity': 1}
improvements = {}
growing = []
successful = []
distribution = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
for country in complete:
    count = 0
    for indicator in directions:
        if changes[country][indicator] * directions[indicator] > 0:
            count += 1
    improvements[country] = count
    if growth[country] > 0:
        growing.append(country)
        distribution[count] += 1
        if count >= 3:
            successful.append(country)
share = len(successful) / len(growing) * 100
if share > 50:
    answer = 'Да'
else:
    answer = 'Нет'
print('Экономик с ростом ВВП на душу:', len(growing))
print('Из них с тремя или четырьмя улучшениями:', len(successful))
print('Доля, %:', round(share, 2))
print('Ответ:', answer)
print('Число улучшений;Число экономик с ростом ВВП')
for count in distribution:
    print(count, distribution[count], sep=';')
print('Экономики с ростом ВВП, но менее чем тремя улучшениями:')
for country in growing:
    if improvements[country] < 3:
        print(names[country], improvements[country], sep=';')

Экономик с ростом ВВП на душу: 150
Из них с тремя или четырьмя улучшениями: 131
Доля, %: 87.33
Ответ: Да
Число улучшений;Число экономик с ростом ВВП
0;0
1;1
2;18
3;77
4;54
Экономики с ростом ВВП, но менее чем тремя улучшениями:
Bahrain;2
Bahamas, The;1
Cuba;2
Cyprus;2
Dominican Republic;2
Fiji;2
France;2
Iceland;2
Italy;2
Korea, Rep.;2
Luxembourg;2
Malaysia;2
Norway;2
Saudi Arabia;2
Somalia, Fed. Rep.;2
Turkmenistan;2
Turkiye;2
Ukraine;2
Samoa;2


## 6. Топ-10 по росту реального ВВП на душу

Рейтинг строится по всем экономикам полной выборки. Относительный прирост лучше абсолютного при сравнении разных начальных уровней. Сортируем пары (−прирост, код) по возрастанию: максимальный прирост будет первым. При равенстве действует алфавитный порядок кода. Рядом выводим число социальных улучшений, чтобы связать рейтинг с исследовательским вопросом.

In [5]:
# Отрицательный прирост позволяет сортировать рейтинг обычным sorted.
rating = []
for country in complete:
    rating.append((-growth[country], country))
rating = sorted(rating)
top10 = rating[:10]
print('Место;Код;Экономика;Прирост ВВП на душу, %;Число социальных улучшений')
for i in range(len(top10)):
    country = top10[i][1]
    print(i + 1, country, names[country], round(growth[country], 3), improvements[country], sep=';')

Место;Код;Экономика;Прирост ВВП на душу, %;Число социальных улучшений
1;CHN;China;79.652;3
2;ETH;Ethiopia;76.512;3
3;KHM;Cambodia;73.752;4
4;TKM;Turkmenistan;72.178;2
5;MMR;Myanmar;68.55;4
6;MNG;Mongolia;66.093;3
7;BGD;Bangladesh;65.759;3
8;LAO;Lao PDR;63.265;3
9;IRL;Ireland;62.764;3
10;GEO;Georgia;61.026;3


## 7. Ответ на вопрос

Да. Из 174 экономик с полными данными у 150 реальный ВВП на душу населения в 2019 году был выше уровня 2010 года. Среди них у 131 улучшились не менее трёх из четырёх социальных показателей: это 87.33%, что превышает 50%. У остальных 19 экономик с ростом ВВП условие не выполнено. Таким образом, по заданному правилу рост выпуска на человека в большинстве наблюдений сопровождался широким социальным улучшением.

Это совместное изменение показателей, а не доказательство, что именно рост ВВП вызвал улучшения. Не учитываются неравенство, качество услуг и промежуточная динамика внутри периода. Из исходных 217 экономик исключены 43 с пропусками: результат относится к полной выборке и может зависеть от доступности данных. Два показателя здоровья связаны, а доступ к электричеству имеет потолок 100%. Условный порог 3 из 4 и одинаковый вес экономик влияют на ответ; статистическая значимость и причинные эффекты не исследуются.

## 8. Запись результатов

Отчёт создаётся через open в режиме w. Он включает вопрос и критерий, состав выборки и причины исключений, таблицы статистик, все изменения, крайние случаи, распределение улучшений, рейтинг и вывод. Повторный запуск перезаписывает только файл результатов.

In [6]:
# Строки отчёта собираем один раз, затем последовательно записываем в файл.
report = []
report.append('ЭКОНОМИЧЕСКИЙ РОСТ И УСЛОВИЯ ЖИЗНИ, 2010–2019')
report.append('Источник: World Bank WDI; данные получены 05.10.2026; CC BY 4.0.')
report.append('Вопрос: у большинства экономик с ростом ВВП на душу улучшились хотя бы 3 из 4 социальных показателей?')
report.append('Улучшения: жизнь и электричество выросли, младенческая смертность и безработица снизились. Равенство не считается улучшением.')
report.append('Да, если доля экономик с 3–4 улучшениями среди экономик с ростом ВВП строго больше 50%; иначе нет.')
report.append(f'Исходных экономик: {len(data)}; полных: {len(complete)}; исключённых: {len(excluded)}.')
report.append('Код исключённой экономики;Название;Пропущенные год:показатель')
for country in excluded:
    report.append(f'{country};{names[country]};{excluded[country]}')
report.append('Единицы: gdp_pc — доллары США 2015 г.; life — лет; infant — на 1000 живорождений; unemployment — % рабочей силы; electricity — % населения.')
report.append('Год;Показатель;Среднее;Медиана;Минимум;Максимум;Размах')
for year in years:
    for indicator in indicators:
        row = stats[year][indicator]
        report.append(f'{year};{indicator};{round(row[0], 4)};{round(row[1], 4)};{round(row[2], 4)};{round(row[3], 4)};{round(row[4], 4)}')
report.append('Показатель;Минимальная Δ;Экономики минимума;Максимальная Δ;Экономики максимума')
for indicator in indicators:
    row = extremes[indicator]
    report.append(f'{indicator};{round(row[0], 4)};{", ".join(row[1])};{round(row[2], 4)};{", ".join(row[3])}')
report.append('Код;Экономика;Δ ВВП на душу, долл. 2015;Прирост ВВП, %;Δ жизни, лет;Δ смертности на 1000;Δ безработицы, п.п.;Δ электричества, п.п.;Социальных улучшений')
for country in complete:
    row = changes[country]
    report.append(f'{country};{names[country]};{round(row["gdp_pc"], 4)};{round(growth[country], 4)};{round(row["life"], 4)};{round(row["infant"], 4)};{round(row["unemployment"], 4)};{round(row["electricity"], 4)};{improvements[country]}')
report.append('Число улучшений;Число экономик с ростом ВВП')
for count in distribution:
    report.append(f'{count};{distribution[count]}')
report.append('Место;Код;Экономика;Прирост ВВП на душу, %;Социальных улучшений')
for i in range(len(top10)):
    country = top10[i][1]
    report.append(f'{i + 1};{country};{names[country]};{round(growth[country], 4)};{improvements[country]}')
report.append(f'Ответ: {answer}. Из {len(growing)} экономик с ростом ВВП у {len(successful)} есть минимум 3 улучшения; доля {round(share, 2)}%.')
report.append('Вывод: в большинстве наблюдений экономический рост сопровождался улучшениями по заданному правилу. Это не доказывает причинную связь.')
report.append('Ограничения: пропуски, условность порога 3 из 4, связь показателей здоровья, потолок электричества 100%, отсутствие весов по населению.')
with open('economy_results.txt', 'w', encoding='utf-8') as output_file:
    for line in report:
        print(line, file=output_file)
print('Создан файл economy_results.txt')

Создан файл economy_results.txt


## 9. Распределение работы

Гусейнов Елшан — 50%: подготовка файла данных; чтение CSV и организация словарей; отбор полной выборки и учёт пропусков; функция расчёта пяти статистик; сравнение двух лет и определение крайних изменений.

Мария Москаленко — 50%: подсчёт социальных улучшений и проверка исследовательского вопроса; построение топ-10; запись результатов в текстовый файл; интерпретация расчётов и оформление PDF и блокнота.

Совместно: постановка вопроса, выбор и обоснование пяти показателей, проверка запуска и согласованности итоговых результатов. Общий вклад: 50% / 50%.